# CMS Medicare Geographic Variation — Data Exploration
**Project:** P1 — CMS Claims & Health Equity Analysis  
**Data:** 2014–2023 Medicare Fee-for-Service Geographic Variation Public Use File (CMS)  
**Goal:** Explore county-level variation in dual eligibility, utilization, and spending to identify health equity patterns across the U.S.

---

In [ ]:
# Import pandas for data loading and manipulation
import pandas as pd

# Load the full CMS Medicare Geographic Variation dataset (2014–2023)
# This file contains national, state, and county-level Medicare FFS metrics
df = pd.read_csv(r'C:\Users\ibrah\Downloads\Medicare Geographic Variation - by National, State & County\Medicare Geographic Variation - by National, State & County\2023\2014-2023 Medicare Fee-for-Service Geographic Variation Public Use File.csv')

# Quick sanity check: confirm row/column count and preview column names
print(df.shape)
print(df.columns.tolist())

## 1. Dataset Structure

In [ ]:
# Summarize dataset dimensions and first few column names
# 247 columns span demographics, spending, utilization, and chronic condition indicators
print("Shape:", df.shape)
print("\nFirst 5 columns:", df.columns[:5].tolist())
print("Total columns:", len(df.columns))

## 2. Geographic Levels & Year Range

In [ ]:
# The dataset includes national, state, and county rows mixed together
# Confirm what geo levels are available before filtering
print(df['BENE_GEO_LVL'].value_counts())

# Confirm which years are available — we'll filter to 2023 for cross-sectional analysis
print("\nYears:", df['YEAR'].unique())

# County rows are our unit of analysis for the health equity work
county_df = df[df['BENE_GEO_LVL'] == 'County']
print("\nCounty rows:", len(county_df))

## 3. Filter to County-Level, 2023

In [ ]:
# Isolate county-level rows for 2023 — our primary unit of analysis
# Using 2023 as a single cross-sectional year avoids temporal confounding
county_2023 = df[(df['BENE_GEO_LVL'] == 'County') & (df['YEAR'] == 2023)]
print("Rows:", len(county_2023))

# Scan for chronic condition columns by keyword — confirming what's available before column selection
cols = df.columns.tolist()
diabetes_cols = [c for c in cols if 'DIAB' in c.upper()]
depression_cols = [c for c in cols if 'DEPR' in c.upper()]
spending_cols = [c for c in cols if 'SPEND' in c.upper() or 'PYMT' in c.upper()]

print("\nDiabetes columns:", diabetes_cols)
print("\nDepression columns:", depression_cols)
print("\nSpending columns:", spending_cols[:10])

In [ ]:
# Broaden the keyword search to surface all chronic condition / prevalence columns
# These could serve as equity indicators or model features in later analysis
chronic_cols = [c for c in cols if any(term in c.upper() for term in 
               ['CHRON', 'CC_', 'COND', 'PREV', 'PCT', 'DIAB', 'DEPR', 'MENTAL', 'ENDOC'])]

print("Chronic/condition columns:", chronic_cols)

In [ ]:
# Print all column names with their index position
# Useful for identifying target columns by number when the dataset is too wide to scan visually
for i, col in enumerate(cols):
    print(i, col)

## 4. Age Level Check

In [ ]:
# BENE_AGE_LVL groups beneficiaries into age brackets rather than raw ages
# Checking distribution and nulls before deciding whether to use it as a filter or feature
print(county_2023['BENE_AGE_LVL'].value_counts())
print(county_2023['BENE_AGE_LVL'].head())
print(county_2023['BENE_AGE_LVL'].isnull().sum())

## 5. Prevention Quality Indicators (PQI)

In [ ]:
# PQIs (AHRQ) measure potentially preventable hospitalizations
# High PQI rates signal gaps in primary/ambulatory care — a key health equity metric
# Check null rates first; CMS suppresses small-cell counts which will show as NaN after type conversion
pqi_cols = [c for c in cols if 'PQI' in c]
print(county_2023[pqi_cols].isnull().sum())
print("\nSample values:")
print(county_2023[pqi_cols].head(3))

## 6. Column Selection — Working Dataset

In [ ]:
# Narrow the dataset to equity indicators, utilization outcomes, and spending metrics
# - BENE_DUAL_PCT: share of Medicare+Medicaid dual-eligible beneficiaries (poverty proxy)
# - Race columns: demographic composition for disparity analysis
# - ER/readmission metrics: utilization intensity (key equity outcomes)
# - FQHC/home health: access-to-care indicators in underserved areas
# - Standardized payments: cost burden adjusted for local price differences
keep_cols = [
    'BENE_GEO_CD', 'BENE_GEO_DESC',
    'BENE_DUAL_PCT', 'BENE_RACE_BLACK_PCT', 
    'BENE_RACE_HSPNC_PCT', 'BENE_RACE_WHT_PCT',
    'BENE_AVG_RISK_SCRE',
    'BENES_ER_VISITS_PCT', 'ER_VISITS_PER_1000_BENES',
    'ACUTE_HOSP_READMSN_PCT',
    'BENES_HH_PCT', 'FQHC_RHC_VISITS_PER_1000_BENES',
    'TOT_MDCR_STDZD_PYMT_PC', 'IP_MDCR_STDZD_PYMT_PC'
]

working_df = county_2023[keep_cols].copy()
print(working_df.shape)
print("\nNull counts:")
print(working_df.isnull().sum())

## 7. Descriptive Statistics

In [ ]:
# Summary statistics across all working columns
# Note any columns with high null rates — CMS suppresses values for counties with < 11 beneficiaries
print(working_df.describe().round(2))

In [ ]:
# Confirm data types — most columns should be numeric
# Object dtype signals string values (e.g. CMS suppression marker '*') that need conversion
print(working_df.dtypes)

In [ ]:
# Inspect raw unique values to confirm what non-numeric entries look like
# CMS uses '*' to suppress small cell counts — these will become NaN after coercion
print(working_df['BENE_DUAL_PCT'].unique()[:20])
print(working_df['TOT_MDCR_STDZD_PYMT_PC'].unique()[:20])

In [ ]:
# Keep geography columns as strings (FIPS code and county name)
# Convert all other columns to numeric — errors='coerce' turns suppressed '*' values into NaN
working_df['BENE_GEO_CD'] = working_df['BENE_GEO_CD'].astype(str)
cols_to_convert = [col for col in working_df.columns
                      if col not in ['BENE_GEO_CD', 'BENE_GEO_DESC']]
working_df[cols_to_convert] = working_df[cols_to_convert].apply(pd.to_numeric, errors='coerce')

# Verify types and check how many NaNs were introduced by suppression
print(working_df.dtypes)
print('\nNull counts after type conversion:')
print(working_df.isnull().sum())
print(working_df.describe().round(2))

## 8. Correlation — Dual Eligibility vs. Utilization & Spending

In [ ]:
# Outcome columns: utilization intensity, spending, and overall health burden
outcome_cols = [
    'BENES_ER_VISITS_PCT',
    'ER_VISITS_PER_1000_BENES',
    'ACUTE_HOSP_READMSN_PCT',
    'TOT_MDCR_STDZD_PYMT_PC',
    'IP_MDCR_STDZD_PYMT_PC',
    'BENE_AVG_RISK_SCRE'
]

# Compute Pearson correlation of each outcome against dual eligibility rate
# A positive correlation means counties with more low-income beneficiaries show higher utilization/spending
correlations = working_df[outcome_cols].corrwith(working_df['BENE_DUAL_PCT'])
print(correlations.round(2))

## 9. Quartile Analysis — Dual Eligibility vs. Outcomes

In [ ]:
# Divide counties into 4 equal-sized groups by dual eligibility rate
# qcut assigns groups by percentile (equal row count), not equal value range
working_df['DUAL_QUARTILE'] = pd.qcut(
    working_df['BENE_DUAL_PCT'],
    q=4,
    labels=('low', 'low-mid', 'high_mid', 'high')
)

# Compute the mean of each outcome column within each dual eligibility quartile
# This lets us compare utilization and spending across the poverty spectrum
quartile_summary = working_df.groupby('DUAL_QUARTILE', observed=True)[[
    'BENES_ER_VISITS_PCT',
    'ER_VISITS_PER_1000_BENES',
    'ACUTE_HOSP_READMSN_PCT',
    'TOT_MDCR_STDZD_PYMT_PC',
    'IP_MDCR_STDZD_PYMT_PC',
    'BENE_AVG_RISK_SCRE'
]].mean().round(2)
print(quartile_summary)

## 10. Magnitude of Change — Low to High Dual Eligibility

In [ ]:
# Calculate percent change in each outcome from the lowest to highest dual eligibility quartile
# This quantifies how much outcomes worsen as poverty concentration increases
low = quartile_summary.loc['low']
high = quartile_summary.loc['high']
pct_change = ((high - low) / low) * 100
print(pct_change)

> **Key Finding:** As dual eligibility increases across county quartiles, ER visit intensity grows nearly 25% while overall health burden (avg risk score) grows only 12%. This suggests low-income Medicare populations are accessing care through emergency channels at rates disproportionate to their underlying health needs — a pattern consistent with inadequate primary care access.

## 11. Visualizations

In [ ]:
# Load visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded successfully')

In [ ]:
# Bar chart: average ER visit rate by dual eligibility quartile
# Shows whether counties with more low-income beneficiaries use the ER more heavily
plt.figure(figsize=(8, 5))
sns.barplot(
    x=quartile_summary.index,
    y=quartile_summary['ER_VISITS_PER_1000_BENES'],
    palette='Blues_d'
)
plt.title('ER Visit Intensity by Dual Eligibility Quartile')
plt.xlabel('Dual Eligibility Quartile')
plt.ylabel('ER Visits per 1,000 Beneficiaries')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: raw relationship between dual eligibility rate and inpatient spending
# Each point is a county — alpha=0.4 reduces overplotting in dense areas
plt.figure(figsize=(9, 6))
sns.scatterplot(
    x='BENE_DUAL_PCT',
    y='IP_MDCR_STDZD_PYMT_PC',
    data=working_df,
    alpha=0.4,
    color='steelblue'
)
plt.title('Dual Eligibility vs Inpatient Spending by County (Raw)')
plt.xlabel('Dual Eligibility Rate')
plt.ylabel('Standardized Inpatient Spending per Capita ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Regression plot: adds a trend line to the scatter to visualize direction and strength
# Red line = OLS fit; confirms whether the dual eligibility / spending relationship is positive
plt.figure(figsize=(9, 6))
sns.regplot(
    x='BENE_DUAL_PCT',
    y='IP_MDCR_STDZD_PYMT_PC',
    data=working_df,
    scatter_kws={'alpha': 0.4, 'color': 'steelblue'},
    line_kws={'color': 'red', 'linewidth': 2}
)
plt.title('Dual Eligibility vs Inpatient Spending by County')
plt.xlabel('Dual Eligibility Rate')
plt.ylabel('Standardized Inpatient Spending per Capita ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart: percent change in each outcome from the lowest to highest dual eligibility quartile
# Normalizes outcomes onto the same scale for direct comparison across metrics
plt.figure(figsize=(8, 5))
sns.barplot(
    x=pct_change.index,
    y=pct_change
)
plt.title('Percent Change in Outcomes from Low to High Dual Eligibility Quartile')
plt.xlabel('Outcome Metric')
plt.ylabel('Percent Change (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()